# Reranking and Late Interaction: Cross-Encoders to ColBERT

The runnable code of [Reranking and Late Interaction: Cross-Encoders to ColBERT](https://residual-stream.lvtan71.workers.dev/posts/reranking-and-late-interaction/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/reranking-and-late-interaction.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 35 min end to end.
The post's run read a copy of the blog taken mid-session, with a draft of post 36 that isn't in git; the rebuild post 37 uses has 308 English sections where the post had 309, so the last digits can differ.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "reranking-and-late-interaction"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/semantic-and-hybrid-search/retrieval.py', 'shared/semantic-and-hybrid-search/sections.py', 'data/blog-post37.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Bi-encoders and cross-encoders

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/reranking-and-late-interaction/#bi-encoders-and-cross-encoders)

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from sections import sections

# Cost per query of the three designs, for this blog's 309 English
# sections, with a MiniLM-sized encoder: 22.7M parameters, about 11M of
# them the token-embedding table (a lookup), so roughly 2 x 11M FLOPs per
# token for the rest, ignoring attention. Queries are the section headings.
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
secs = sections("en")
full = np.array([len(tok(s["body"]).input_ids) for s in secs])
read = np.minimum(full, 512)                                   # what a 512-token encoder reads
q = np.mean([len(tok(s["heading"]).input_ids) for s in secs])
N, flop_tok, K = len(secs), 2 * 11e6, 50
print(f"{N} sections, {full.mean():.0f} tokens each on average ({read.mean():.0f} read at most 512); "
      f"queries average {q:.1f} tokens")
bi = q * flop_tok + N * 384 * 2
cross = K * (q + read.mean()) * flop_tok
late = q * flop_tok + N * q * full.mean() * 128 * 2
print(f"bi-encoder: encode the query, then {N} dot products           {bi / 1e6:10.1f} MFLOPs per query")
print(f"cross-encoder, top {K}: {K} passes over query + section        {cross / 1e6:10.1f} MFLOPs per query")
print(f"late interaction (ColBERT, 128-d): every query x section token {late / 1e6:10.1f} MFLOPs per query")
print(f"  roughly K (q + L) / q = {K * (q + read.mean()) / q:,.0f}: the cross-encoder's multiple of one query encoding")
for n_docs in (1e4, 1e6):
    print(f"  at {n_docs:,.0f} documents: bi-encoder {(q * flop_tok + n_docs * 384 * 2) / 1e9:6.2f} GFLOPs, "
          f"MaxSim over every document {(q * flop_tok + n_docs * q * full.mean() * 256) / 1e9:8.1f} GFLOPs")
print("\nindex size, every token indexed, 16-bit numbers:")
print(f"  bi-encoder, one 384-d vector per section:              {N * 384 * 2 / 2**20:7.2f} MiB")
print(f"  ColBERT, one 128-d vector per token:                   {full.sum() * 128 * 2 / 2**20:7.2f} MiB")
for b in (2, 1):
    print(f"  ColBERTv2, 4-byte centroid id + {b}-bit residuals:       {full.sum() * (4 + 128 * b / 8) / 2**20:7.2f} MiB")

The post printed:

```text
309 sections, 426 tokens each on average (361 read at most 512); queries average 9.0 tokens
bi-encoder: encode the query, then 309 dot products                198.3 MFLOPs per query
cross-encoder, top 50: 50 passes over query + section          407149.5 MFLOPs per query
late interaction (ColBERT, 128-d): every query x section token      501.3 MFLOPs per query
  roughly K (q + L) / q = 2,056: the cross-encoder's multiple of one query encoding
  at 10,000 documents: bi-encoder   0.21 GFLOPs, MaxSim over every document     10.0 GFLOPs
  at 1,000,000 documents: bi-encoder   0.97 GFLOPs, MaxSim over every document    981.6 GFLOPs

index size, every token indexed, 16-bit numbers:
  bi-encoder, one 384-d vector per section:                 0.23 MiB
  ColBERT, one 128-d vector per token:                     32.12 MiB
  ColBERTv2, 4-byte centroid id + 2-bit residuals:          4.52 MiB
  ColBERTv2, 4-byte centroid id + 1-bit residuals:          2.51 MiB
```

## Late interaction: ColBERT

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/reranking-and-late-interaction/#late-interaction-colbert)

In [ ]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

# Late interaction (ColBERT's MaxSim), worked on a 3-word query (4 word pieces). Every
# token of the query and of each document gets its own vector; each query
# token takes its best match among the document's tokens, and the score is
# the sum of those maxima. The token vectors here are all-MiniLM-L6-v2's,
# which was not trained for this: the arithmetic is ColBERT's, the numbers
# only illustrate it.
path = "sentence-transformers/all-MiniLM-L6-v2"
tok, enc = AutoTokenizer.from_pretrained(path), AutoModel.from_pretrained(path).eval()
def vectors(text):
    ids = tok(text, return_tensors="pt")
    with torch.no_grad():
        h = enc(**ids).last_hidden_state[0, 1:-1]                     # drop [CLS] and [SEP]
    return tok.convert_ids_to_tokens(ids.input_ids[0, 1:-1]), torch.nn.functional.normalize(h, dim=-1).numpy()

q_tokens, Qv = vectors("paged kv cache")
for doc in ("vllm stores the kv cache in fixed-size pages, like virtual memory",
            "the cache line of a cpu holds sixty-four bytes"):
    d_tokens, Dv = vectors(doc)
    S = Qv @ Dv.T                                                     # every query token vs every document token
    best = S.argmax(1)
    print(f'document: "{doc}"')
    for i, t in enumerate(q_tokens):
        print(f"  {t:7s} best match {d_tokens[best[i]]!r:12s} {S[i, best[i]]:.2f}   (mean over the document {S[i].mean():.2f})")
    print(f"  MaxSim score {S.max(1).sum():.2f}; mean-of-similarities score {S.mean(1).sum():.2f}")

The post printed:

```text
document: "vllm stores the kv cache in fixed-size pages, like virtual memory"
  page    best match 'pages'      0.75   (mean over the document 0.29)
  ##d     best match 'like'       0.46   (mean over the document 0.30)
  kv      best match 'kv'         0.92   (mean over the document 0.30)
  cache   best match 'cache'      0.92   (mean over the document 0.35)
  MaxSim score 3.06; mean-of-similarities score 1.24
document: "the cache line of a cpu holds sixty-four bytes"
  page    best match 'cache'      0.35   (mean over the document 0.22)
  ##d     best match 'cache'      0.36   (mean over the document 0.24)
  kv      best match 'cpu'        0.24   (mean over the document 0.14)
  cache   best match 'cache'      0.87   (mean over the document 0.35)
  MaxSim score 1.81; mean-of-similarities score 0.95
```

## Measured: reranking the benchmark from Semantic and Hybrid Search

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/reranking-and-late-interaction/#measured-reranking-the-benchmark-from-semantic-and-hybrid-search)

In [ ]:
import time
import numpy as np
from sentence_transformers import CrossEncoder
exec(open("retrieval.py").read().split("# How much each encoder reads")[0])   # data from "Semantic and Hybrid Search", BM25, dense, RRF

# Two-stage retrieval on the benchmark from "Semantic and Hybrid Search". Stage 1, cheap, over every
# section: the best ranker from "Semantic and Hybrid Search" for the language (English: BM25 fused with
# dense passages; Vietnamese: BM25). Stage 2, expensive, over its top 50
# only: a cross-encoder reads query and section together and scores the
# pair (English: ms-marco-MiniLM-L6-v2; multilingual:
# mmarco-mMiniLMv2-L12-H384-v1). Sections it ranks are cut to its input
# limit, 512 tokens; one run instead scores 80-word passages and keeps each
# section's best.
K = 50
def two_stage(title, corpus, queries, stage1, ce_name, passages=False):
    ce = CrossEncoder(ce_name, device="cpu")
    docs = [s["body"] for s in corpus]
    before, after, seconds = [], [], []
    for q, target in queries:
        s1 = stage1(q)
        top = np.argsort(-s1)[:K]
        before.append(int((s1 >= s1[target]).sum()))
        t0 = time.perf_counter()
        if passages:                                                     # score each 80-word passage, keep the best
            pairs, owner = [], []
            for j, i in enumerate(top):
                w = docs[i].split()
                for p in range(0, len(w), 80):
                    pairs.append((q, " ".join(w[p:p + 80]))); owner.append(j)
            sc = ce.predict(pairs, batch_size=64)
            s2 = np.full(K, -np.inf); np.maximum.at(s2, np.array(owner), sc)
        else:
            s2 = ce.predict([(q, docs[i]) for i in top], batch_size=K)
        seconds.append(time.perf_counter() - t0)
        if target in top:
            after.append(int((s2 >= s2[list(top).index(target)]).sum()))
        else:
            after.append(before[-1])                                     # stage 1 missed it; rerank can't help
    b, a = np.array(before), np.array(after)
    d = 1 / a - 1 / b
    rng = np.random.default_rng(0)
    lo, hi = np.percentile([d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)], [2.5, 97.5])
    print(f"{title} ({len(queries)} queries), cross-encoder {ce_name.split('/')[1]}"
          f"{', best 80-word passage' if passages else ''}:")
    print(f"  stage 1 alone:    recall@1 {np.mean(b == 1):4.0%}  MRR {np.mean(1 / b):.3f}  (right section in its top {K}: {np.mean(b <= K):.0%})")
    print(f"  + re-rank top {K}: recall@1 {np.mean(a == 1):4.0%}  MRR {np.mean(1 / a):.3f}  "
          f"difference {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")
    print(f"  re-ranking time: {np.median(seconds) * 1e3:.0f} ms per query on the CPU ({K} pairs)")

en_docs, vi_docs = [s["body"] for s in en], [s["body"] for s in vi]
en_q = [(s["heading"], i) for i, s in enumerate(en) if s in unique(en)]
vi_q = [(s["heading"], i) for i, s in enumerate(vi) if s in unique(vi)]
en_stage1 = rrf(BM25(en_docs).scores, dense_passages("MiniLM (English)", en_docs))
vi_stage1 = BM25(vi_docs).scores
two_stage("English", en, en_q, en_stage1, "cross-encoder/ms-marco-MiniLM-L6-v2")
two_stage("English", en, en_q, en_stage1, "cross-encoder/ms-marco-MiniLM-L6-v2", passages=True)
two_stage("English", en, en_q, en_stage1, "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
two_stage("Vietnamese", vi, vi_q, vi_stage1, "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
two_stage("Vietnamese", vi, vi_q, vi_stage1, "cross-encoder/ms-marco-MiniLM-L6-v2")

The post printed:

```text
English (297 queries), cross-encoder ms-marco-MiniLM-L6-v2:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  45%  MRR 0.575  difference +0.045 [+0.012, +0.079]
  re-ranking time: 1468 ms per query on the CPU (50 pairs)
English (297 queries), cross-encoder ms-marco-MiniLM-L6-v2, best 80-word passage:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  40%  MRR 0.535  difference +0.004 [-0.032, +0.040]
  re-ranking time: 1737 ms per query on the CPU (50 pairs)
English (297 queries), cross-encoder mmarco-mMiniLMv2-L12-H384-v1:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  39%  MRR 0.516  difference -0.014 [-0.054, +0.025]
  re-ranking time: 2377 ms per query on the CPU (50 pairs)
Vietnamese (257 queries), cross-encoder mmarco-mMiniLMv2-L12-H384-v1:
  stage 1 alone:    recall@1  42%  MRR 0.539  (right section in its top 50: 91%)
  + re-rank top 50: recall@1  41%  MRR 0.529  difference -0.010 [-0.055, +0.037]
  re-ranking time: 2317 ms per query on the CPU (50 pairs)
Vietnamese (257 queries), cross-encoder ms-marco-MiniLM-L6-v2:
  stage 1 alone:    recall@1  42%  MRR 0.539  (right section in its top 50: 91%)
  + re-rank top 50: recall@1  35%  MRR 0.487  difference -0.053 [-0.098, -0.007]
  re-ranking time: 976 ms per query on the CPU (50 pairs)
```